# Redis-based Airbnb Data Management for New York City

**Course:** Business Intelligence & Big Data Analytics   
**University:** University of Milano-Bicocca  

**Team Members:**  
- Aarohi Mistry (925352)  
- Any Das (922710)  

---

### **Project Goal**

Design and implement an automated pipeline that:  
1. Downloads New York Airbnb public datasets directly from Inside Airbnb.  
2. Loads and processes the data using Python.  
3. Stores relevant information in Redis using an efficient data model.  
4. Allows fast querying for insights useful to different stakeholders.  

### **Step 1: Install Required Libraries & Start Redis Server**

In [ ]:
# Install redis server & required libraries
!apt-get -y install redis-server
!pip install -q redis pandas matplotlib

# Start redis server in background
!redis-server --daemonize yes

# Test Redis server
!redis-cli ping

print("Libraries installed & Redis server started successfully!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-tools
Suggested packages:
  ruby-redis
The following NEW packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-server
  redis-tools
0 upgraded, 7 newly installed, 0 to remove and 1 not upgraded.
Need to get 1,273 kB of archives.
After this operation, 5,725 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjemalloc2 amd64 5.2.1-4ubuntu1 [240 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblua5.1-0 amd64 5.1.5-8.1build4 [99.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblzf1 amd64 3.6-3 [7,444 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 lua-bitop amd64 1.0.2-5 [6,680 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 lua-

### **Step 2: Download Airbnb New York City Data**

Files to download:
- `listings.csv.gz` → Details of each Airbnb listing  
- `calendar.csv.gz` → Price & availability per date  
- `reviews.csv.gz` → Reviews posted by guests    

We use `.gz` compressed files because they contain full data with all attributes.


In [ ]:
LISTINGS_URL = "https://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/listings.csv.gz"
CALENDAR_URL = "https://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/calendar.csv.gz"
REVIEWS_URL  = "https://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/reviews.csv.gz"

!wget -O listings.csv.gz "$LISTINGS_URL"
!wget -O calendar.csv.gz "$CALENDAR_URL"
!wget -O reviews.csv.gz  "$REVIEWS_URL"

print("\nDownload complete: listings.csv.gz | calendar.csv.gz | reviews.csv.gz")

--2025-12-22 15:16:09--  https://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/listings.csv.gz
Resolving data.insideairbnb.com (data.insideairbnb.com)... 13.33.67.90, 13.33.67.71, 13.33.67.116, ...
Connecting to data.insideairbnb.com (data.insideairbnb.com)|13.33.67.90|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17748368 (17M) [application/x-gzip]
Saving to: ‘listings.csv.gz’

listings.csv.gz     100%[===================>]  16.93M  26.1MB/s    in 0.6s    

2025-12-22 15:16:10 (26.1 MB/s) - ‘listings.csv.gz’ saved [17748368/17748368]

--2025-12-22 15:16:10--  https://data.insideairbnb.com/united-states/ny/new-york-city/2025-10-01/data/calendar.csv.gz
Resolving data.insideairbnb.com (data.insideairbnb.com)... 13.33.67.90, 13.33.67.71, 13.33.67.116, ...
Connecting to data.insideairbnb.com (data.insideairbnb.com)|13.33.67.90|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31060532 (30M) [application/x-gzip]
Savi

### **Step 3: Load Datasets using Pandas**

We now load the three `.csv.gz` files into pandas DataFrames.

In [ ]:
import pandas as pd

# Load compressed CSV files
listings = pd.read_csv("listings.csv.gz", compression="gzip", low_memory=False)
calendar = pd.read_csv("calendar.csv.gz", compression="gzip", low_memory=False)
reviews  = pd.read_csv("reviews.csv.gz",  compression="gzip", low_memory=False)

print("Listings rows:", len(listings))
print("Calendar rows:", len(calendar))
print("Reviews rows:", len(reviews))

print("\n--- Listings sample ---")
display(listings.head(3))

print("\n--- Calendar sample ---")
display(calendar.head(3))

print("\n--- Reviews sample ---")
display(reviews.head(3))

Listings rows: 36111
Calendar rows: 13180517
Reviews rows: 986597

--- Listings sample ---


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,40824219,https://www.airbnb.com/rooms/40824219,20251001171547,2025-10-02,city scrape,Room close to Manhattan for FEMALE guests,This cozy spacious room includes a twin size b...,Sunnyside is a safe residental area. <br />The...,https://a0.muscache.com/pictures/hosting/Hosti...,317540555,...,4.88,4.94,4.69,NaN,f,3,0,3,0,0.23
1,40833186,https://www.airbnb.com/rooms/40833186,20251001171547,2025-10-02,previous scrape,Soho LES East village private room downtown,NaN,NaN,https://a0.muscache.com/pictures/1f093bbc-936c...,68718914,...,NaN,NaN,NaN,NaN,t,1,0,1,0,NaN
2,40837137,https://www.airbnb.com/rooms/40837137,20251001171547,2025-10-02,previous scrape,Sunset Park - Quiet and close to subway!,"Cozy, lovely bedroom with a comfortable full s...",the sunset park of Brooklyn,https://a0.muscache.com/pictures/01c4e91e-4012...,317770098,...,5.00,5.00,5.00,NaN,f,1,0,1,0,0.01



--- Calendar sample ---


,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,40824219,2025-10-02,f,NaN,NaN,30,365
1,40824219,2025-10-03,f,NaN,NaN,30,365
2,40824219,2025-10-04,f,NaN,NaN,30,365



--- Reviews sample ---


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2595,17857,2009-11-21,50679,Jean,Notre séjour de trois nuits.\r<br/>Nous avons ...
1,2595,19176,2009-12-05,53267,Cate,Great experience.
2,2595,46312,2010-05-25,117113,Alicia,We had a wonderful stay at Jennifer's charming...


In [ ]:
listings.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [ ]:
calendar.columns

Index(['listing_id', 'date', 'available', 'price', 'adjusted_price',
       'minimum_nights', 'maximum_nights'],
      dtype='object')

In [ ]:
reviews.columns

Index(['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments'], dtype='object')

### **Step 4: Clean and Prepare Listing Data**

From the full `listings` dataset (79 columns), we only keep the fields that are most relevant for our Redis data model and business queries.

We keep:
- `id` – unique listing identifier  
- `name` – title of the listing  
- `host_id` – owner/host identifier  
- `neighbourhood_cleansed` – cleaned neighbourhood name  
- `room_type` – type of room (entire, private, shared)  
- `price` – nightly price  
- `number_of_reviews` – total number of reviews  
- `review_scores_rating` – average rating score  
- `latitude`, `longitude` – listing coordinates (useful for geo queries)

We also clean the `price` field (remove `$` and commas) and handle missing ratings.


In [ ]:
# ---------------------- STEP 4: CLEAN & PREPARE LISTINGS ---------------------- #

# 1) Select only the important columns from listings.csv
important_cols = [
    "id", "name", "host_id", "neighbourhood_cleansed",
    "room_type", "price", "number_of_reviews",
    "review_scores_rating", "latitude", "longitude"
]

listings_small = listings[important_cols].copy()

# 2) Clean the price column in listings.csv (remove $, commas → convert to number)
listings_small["price"] = (
    listings_small["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
listings_small["price"] = pd.to_numeric(listings_small["price"], errors="coerce")

# ---------------------- USE CALENDAR DATA TO FILL MISSING PRICE ---------------------- #

# Clean the calendar price column
calendar_price_raw = calendar["price"]

# If the dataset has an "adjusted_price" column, use it when price is missing
if "adjusted_price" in calendar.columns:
    calendar_price_raw = calendar_price_raw.fillna(calendar["adjusted_price"])

# Convert calendar price to numeric
calendar_price_clean = (
    calendar_price_raw.astype(str)
    .str.replace("$", "")
    .str.replace(",", "")
)
calendar_price_clean = pd.to_numeric(calendar_price_clean, errors="coerce")

# Add clean price back to calendar df
calendar = calendar.copy()
calendar["price_clean"] = calendar_price_clean

# Compute AVERAGE daily price per listing using calendar.csv
calendar_avg_price = (
    calendar
    .dropna(subset=["price_clean"])
    .groupby("listing_id")["price_clean"]
    .mean()
    .reset_index()
    .rename(columns={"price_clean": "avg_calendar_price"})
)

# Merge this average price into listings_small
listings_small = listings_small.merge(
    calendar_avg_price,
    left_on="id",
    right_on="listing_id",
    how="left"
)

# Fill missing prices using calendar averages
listings_small["price"] = listings_small["price"].fillna(listings_small["avg_calendar_price"])

# Remove helper columns after merge
listings_small.drop(columns=["listing_id", "avg_calendar_price"], inplace=True)

# ---------------------- FALLBACK: FILL REMAINING MISSING PRICE ---------------------- #

# If price is still missing, fill using neighbourhood + room_type median
listings_small["neighbourhood_cleansed"] = listings_small["neighbourhood_cleansed"].fillna("Unknown")
listings_small["room_type"] = listings_small["room_type"].fillna("Unknown")

listings_small["price"] = (
    listings_small
    .groupby(["neighbourhood_cleansed", "room_type"])["price"]
    .transform(lambda x: x.fillna(x.median()))
)

# Final fallback: use global median (very rare missing cases)
global_median = listings_small["price"].median()
listings_small["price"] = listings_small["price"].fillna(global_median)

# ---------------------- FIX MISSING NAME VALUES ---------------------- #

# Replace missing names with a simple label
listings_small["name"] = listings_small["name"].fillna("Unknown Listing")

# ---------------------- FINAL CLEANUP ---------------------- #

listings_small["number_of_reviews"] = listings_small["number_of_reviews"].fillna(0).astype(int)
listings_small["review_scores_rating"] = listings_small["review_scores_rating"].fillna(0)

print("Cleaned listings preview:")
display(listings_small.head())

print("\nMissing value stats:")
print(listings_small.isna().sum())


Cleaned listings preview:


,id,name,host_id,neighbourhood_cleansed,room_type,price,number_of_reviews,review_scores_rating,latitude,longitude
0,40824219,Room close to Manhattan for FEMALE guests,317540555,Sunnyside,Private room,66.0,16,4.81,40.74698,-73.91763
1,40833186,Soho LES East village private room downtown,68718914,Nolita,Private room,168.0,0,0.00,40.72314,-73.99323
2,40837137,Sunset Park - Quiet and close to subway!,317770098,Sunset Park,Private room,88.0,1,5.00,40.64607,-74.00552
3,40838018,Cozy One Bedroom in Clinton Hill,17211451,Clinton Hill,Entire home/apt,257.0,1,5.00,40.68370,-73.96115
4,40839416,🪴XL dojo 🌾 shared green yogi palace apt 🌿,4765305,East Village,Private room,76.0,20,4.95,40.72147,-73.98270



Missing value stats:
id                        0
name                      0
host_id                   0
neighbourhood_cleansed    0
room_type                 0
price                     0
number_of_reviews         0
review_scores_rating      0
latitude                  0
longitude                 0
dtype: int64


### **Step 5: Insert cleaned listing data into Redis**

For each listing, we:

1. Create a hash: `listing:<id>` with basic attributes.  
2. Add the listing id to:
   - `city:newyork:listings` (set of all listings in New York City)
   - `neighbourhood:<name>:listings` (set per neighbourhood)
3. Add the listing id to a sorted set `nyc:top_rated` with the rating score as the sorting key.
4. Optionally, store coordinates in a Redis GEO index for future geo-based queries.


In [ ]:
import redis, math

print("Connecting to Redis...")
r = redis.Redis(host="localhost", port=6379, db=0)
print("Redis ping:", r.ping())

print("Inserting cleaned listings into Redis...")
count = 0

for idx, row in listings_small.iterrows():
    listing_id = int(row["id"])
    key = f"listing:{listing_id}"

    # Data stored inside hash
    listing_data = {
        "name": str(row["name"]),
        "host_id": str(row["host_id"]),
        "neighbourhood": str(row["neighbourhood_cleansed"]),
        "room_type": str(row["room_type"]),
        "price": str(float(row["price"])),
        "num_reviews": str(int(row["number_of_reviews"])),
        "rating": str(float(row["review_scores_rating"]))
    }

    # 1) Insert hash into Redis
    r.hset(key, mapping=listing_data)

    # 2) Add listing to city-level set
    r.sadd("city:newyork:listings", listing_id)

    # 3) Add listing to neighbourhood-level set
    r.sadd(f"neighbourhood:{listing_data['neighbourhood']}:listings", listing_id)

    # 4) Add to sorted set (for rating rankings)
    r.zadd("nyc:top_rated", {listing_id: float(row["review_scores_rating"])})

    # 5) Add coordinates to GEO index
    lat, lon = row["latitude"], row["longitude"]
    if not math.isnan(lat) and not math.isnan(lon):
        r.geoadd("nyc:listings:geo", (float(lon), float(lat), listing_id))

    count += 1

print(f"\nInserted {count} records into Redis ✔")
print("Total listings in Redis:", r.scard("city:newyork:listings"))


Connecting to Redis...
Redis ping: True
Inserting cleaned listings into Redis...

Inserted 36111 records into Redis ✔
Total listings in Redis: 36111


### **Step 6: Redis Queries (Validation & Ranking)**



### **Step 6.1: Sanity Checks**

We verify that:
- The total number of listings in Redis matches the dataset.
- We can read one random listing from Redis and inspect its fields.

In [ ]:
# 1) Total listings in Redis
total = r.scard("city:newyork:listings")
print("Total listings in Redis (city:newyork:listings):", total)

# 2) Get one random listing id
rand_id = r.srandmember("city:newyork:listings")
print("\nRandom listing ID:", rand_id)

# 3) Show its data
if rand_id is not None:
    key = f"listing:{int(rand_id)}"
    data = r.hgetall(key)

    print("\nRaw hash data for", key, ":")
    print(data)

    print("\nDecoded view:")
    for k, v in data.items():
        print(k.decode(), ":", v.decode())


Total listings in Redis (city:newyork:listings): 36111

Random listing ID: b'53041846'

Raw hash data for listing:53041846 :
{b'name': b'Luxury building, HD TV, gym, laundry, rooftop #187', b'host_id': b'3223938', b'neighbourhood': b'Bushwick', b'room_type': b'Private room', b'price': b'61.0', b'num_reviews': b'1', b'rating': b'5.0'}

Decoded view:
name : Luxury building, HD TV, gym, laundry, rooftop #187
host_id : 3223938
neighbourhood : Bushwick
room_type : Private room
price : 61.0
num_reviews : 1
rating : 5.0


### **Step 6.2: Top Listings Ranking (Rating DESC, Price ASC)**

We query the sorted set `nyc:top_rated` and then apply a secondary sort by price
to display top listings in a readable table (including room type).


In [ ]:
from prettytable import PrettyTable

print("\nTop 10 Highest Rated Listings in NYC (Rating DESC, Price ASC)\n")

table = PrettyTable()
table.field_names = ["Listing ID", "Rating", "Price ($)", "Room Type", "Neighbourhood", "Name (Shortened)"]

# 1) Fetch more than 10 items from Redis to allow second-level sorting
top_raw = r.zrevrange("nyc:top_rated", 0, 99, withscores=True)

rows = []

for lid, score in top_raw:
    lid_int = int(lid)
    data = r.hgetall(f"listing:{lid_int}")

    name = data.get(b"name", b"").decode("utf-8")
    neighbourhood = data.get(b"neighbourhood", b"").decode("utf-8")
    room_type = data.get(b"room_type", b"").decode("utf-8")
    price_str = data.get(b"price", b"0").decode("utf-8")

    # Convert price from string to float
    try:
        price_val = float(price_str)
    except:
        price_val = 9999999   # treat missing / weird price as very high

    rows.append({
        "id": lid_int,
        "rating": float(score),
        "price": price_val,
        "room_type": room_type,
        "neighbourhood": neighbourhood,
        "name": name
    })

# 2) Sort by: rating DESC, price ASC
rows_sorted = sorted(rows, key=lambda x: (-x["rating"], x["price"]))

# 3) Take top 10 & print
for row in rows_sorted[:10]:
    short_name = (row["name"][:40] + "...") if len(row["name"]) > 40 else row["name"]
    table.add_row([
        row["id"],
        f"{row['rating']:.2f}",
        f"{row['price']:.1f}",
        row["room_type"],
        row["neighbourhood"],
        short_name
    ])

print(table)



Top 10 Highest Rated Listings in NYC (Rating DESC, Price ASC)

+--------------------+--------+-----------+--------------+--------------------+---------------------------------------------+
|     Listing ID     | Rating | Price ($) |  Room Type   |   Neighbourhood    |               Name (Shortened)              |
+--------------------+--------+-----------+--------------+--------------------+---------------------------------------------+
| 995268785620684770 |  5.00  |    41.0   | Private room | Bedford-Stuyvesant | Bright bedroom with high ceiling in Beds... |
| 988787058900567282 |  5.00  |    41.0   | Private room | Bedford-Stuyvesant | Luxury Duplex Apt, HDTV, laundry, terrac... |
| 991301559931083002 |  5.00  |    44.0   | Private room |       Harlem       |            Beautiful Private room           |
| 984866964862347137 |  5.00  |    44.0   | Private room |     Bay Ridge      |              stylish (balcony)              |
| 992033931066536338 |  5.00  |    45.0   | Private ro

In [ ]:
from prettytable import PrettyTable

print("\nTop 10 Listings Sorted by: Rating (DESC), Price (DESC)\n")

table = PrettyTable()
table.field_names = ["Listing ID", "Rating", "Price ($)", "Room Type", "Neighbourhood", "Name"]

# Get more items for secondary sorting
top_raw = r.zrevrange("nyc:top_rated", 0, 99, withscores=True)

rows = []

for lid, score in top_raw:
    lid_int = int(lid)
    data = r.hgetall(f"listing:{lid_int}")

    name = data.get(b"name", b"").decode()
    neighbourhood = data.get(b"neighbourhood", b"").decode()
    room_type = data.get(b"room_type", b"").decode()
    price = float(data.get(b"price", b"0").decode())

    rows.append({
        "id": lid_int,
        "rating": float(score),
        "price": price,
        "room_type": room_type,
        "neighbourhood": neighbourhood,
        "name": name
    })

# SORT BY rating (DESC) then price (DESC)
rows_sorted = sorted(rows, key=lambda x: (-x["rating"], -x["price"]))

# Show first 10
for row in rows_sorted[:10]:
    table.add_row([
        row["id"],
        f"{row['rating']:.2f}",
        f"{row['price']:.2f}",
        row["room_type"],
        row["neighbourhood"],
        row["name"][:40] + ("..." if len(row["name"]) > 40 else "")
    ])

print(table)



Top 10 Listings Sorted by: Rating (DESC), Price (DESC)

+--------------------+--------+-----------+-----------------+-----------------+---------------------------------------------+
|     Listing ID     | Rating | Price ($) |    Room Type    |  Neighbourhood  |                     Name                    |
+--------------------+--------+-----------+-----------------+-----------------+---------------------------------------------+
|      9879796       |  5.00  |   629.00  | Entire home/apt | Upper East Side | Modern 2 Bedroom by Memorial Sloan Kette... |
| 990798417345656604 |  5.00  |   614.00  | Entire home/apt |     Midtown     |        Beautiful Central Park studio        |
| 998088361998367565 |  5.00  |   597.00  | Entire home/apt |    Chinatown    | Luxury Penthouse! 2 Bed / 2 Bath + Priva... |
| 986263425862656510 |  5.00  |   588.00  | Entire home/apt |  Hell's Kitchen | Gorgeous space near Hells Kitchen and Mi... |
| 984667555924799835 |  5.00  |   526.00  | Entire home/apt |

### **Step 7: Neighbourhood-level BI Analysis**

In this step, we analyze Airbnb listings at neighbourhood level to understand
supply concentration, pricing patterns, and average listing quality.

In [ ]:
# Neighbourhood-level aggregation using listings data

neigh_stats = (
    listings_small
    .groupby("neighbourhood_cleansed")
    .agg(
        total_listings=("id", "count"),
        avg_price=("price", "mean"),
        avg_rating=("review_scores_rating", "mean")
    )
    .reset_index()
    .sort_values("total_listings", ascending=False)
)

neigh_stats.head(10)


,neighbourhood_cleansed,total_listings,avg_price,avg_rating
13,Bedford-Stuyvesant,2612,132.180513,3.289548
217,Williamsburg,2049,470.362128,3.457706
131,Midtown,1993,2573.406422,2.584124
97,Harlem,1685,155.001484,3.547181
98,Hell's Kitchen,1496,1391.211230,2.982981
205,Upper West Side,1458,1238.795610,2.655501
28,Bushwick,1443,105.699931,3.107277
204,Upper East Side,1399,1051.037884,3.106605
53,Crown Heights,1133,168.517211,3.331253
66,East Village,959,233.087591,3.334025


### **Step 8: Availability and Pricing Analysis using Calendar Data**

In this step, we analyze listing availability and daily prices using the
calendar dataset to better understand demand and pricing dynamics across
New York City neighbourhoods.


In [ ]:
# Convert availability to numeric (1 = available, 0 = booked)
calendar["available_num"] = calendar["available"].map({"t": 1, "f": 0})

# Merge calendar with neighbourhood info
calendar_neigh = calendar.merge(
    listings_small[["id", "neighbourhood_cleansed"]],
    left_on="listing_id",
    right_on="id",
    how="left"
)

# Neighbourhood-level availability stats
availability_stats = (
    calendar_neigh
    .groupby("neighbourhood_cleansed")
    .agg(
        avg_availability=("available_num", "mean")
    )
    .reset_index()
    .sort_values("avg_availability")
)

availability_stats.head(10)

,neighbourhood_cleansed,avg_availability
145,New Dorp,0.000000
201,Two Bridges,0.199087
107,Inwood,0.202690
46,Columbia St,0.204110
134,Morningside Heights,0.204832
120,Little Neck,0.209589
170,Riverdale,0.216895
38,Civic Center,0.222356
6,Battery Park City,0.222706
162,Prospect Heights,0.243682


### **Step 8.2: Convert availability to percentage**


In [ ]:
# Create availability percentage column
availability_stats["availability_pct"] = availability_stats["avg_availability"] * 100

# Round for better readability
availability_stats["availability_pct"] = availability_stats["availability_pct"].round(2)

availability_stats.head(10)


,neighbourhood_cleansed,avg_availability,availability_pct
145,New Dorp,0.000000,0.00
201,Two Bridges,0.199087,19.91
107,Inwood,0.202690,20.27
46,Columbia St,0.204110,20.41
134,Morningside Heights,0.204832,20.48
120,Little Neck,0.209589,20.96
170,Riverdale,0.216895,21.69
38,Civic Center,0.222356,22.24
6,Battery Park City,0.222706,22.27
162,Prospect Heights,0.243682,24.37


### **Step 8.3: Availability by Neighbourhood (Most Booked Areas)**

The following chart shows neighbourhoods with the lowest average availability.
Lower values indicate higher booking frequency and stronger demand.


In [ ]:
import plotly.express as px

most_booked = availability_stats.head(10)

fig = px.bar(
    most_booked,
    x="availability_pct",
    y="neighbourhood_cleansed",
    orientation="h",
    title="Top 10 Most Booked Neighbourhoods in NYC",
    color="avg_availability",
    color_continuous_scale="Blues"
)

fig.update_layout(
    xaxis_title="Availability (%) — Lower = More Booked",
    yaxis_title="Neighbourhood",
    yaxis=dict(autorange="reversed"),
    height=500
)

fig.show()


### **Step 9: Amenities Analysis (Additional Insights)**

In this step, we analyze the amenities offered by highly rated listings
to identify features commonly associated with better guest experiences.
This section provides additional business insights beyond the core analysis.


In [ ]:
# Select listings with high ratings (e.g. rating >= 4.5)
high_rated = listings[listings["review_scores_rating"] >= 4.5].copy()

# Drop rows with missing amenities
high_rated = high_rated.dropna(subset=["amenities"])

print("Number of high-rated listings:", len(high_rated))


Number of high-rated listings: 21367


### **Step 9.2: Extract & count amenities**

In [ ]:
from collections import Counter
import ast

amenity_counter = Counter()

for amenities_str in high_rated["amenities"]:
    try:
        # Convert string representation to list
        amenities_list = ast.literal_eval(amenities_str)
        amenity_counter.update(amenities_list)
    except:
        continue

# Convert to DataFrame
amenities_df = (
    pd.DataFrame(amenity_counter.items(), columns=["amenity", "count"])
    .sort_values("count", ascending=False)
)

amenities_df.head(10)


,amenity,count
12,Smoke alarm,19930
14,Wifi,19671
28,Kitchen,19044
36,Essentials,18032
15,Carbon monoxide alarm,17419
22,Hangers,16643
21,Hot water,16141
29,Hair dryer,15349
4,Iron,14931
19,Heating,14765


### **Step 9.3: Most Common Amenities in Highly Rated Listings**

The chart below shows the most frequently offered amenities among listings
with high guest ratings.


In [ ]:
import plotly.express as px

top_amenities = amenities_df.head(10)

fig = px.bar(
    top_amenities,
    x="count",
    y="amenity",
    orientation="h",
    color="count",
    color_continuous_scale="Blues",
    title="Top 10 Amenities in Highly Rated Listings"
)

fig.update_layout(
    xaxis_title="Number of Listings",
    yaxis_title="Amenity",
    yaxis=dict(autorange="reversed"),
    height=500
)

fig.show()

### **Conclusion**

Overall, the combination of Redis-based data storage and business intelligence
analysis enables fast querying and meaningful insight extraction from large
Airbnb datasets. The project highlights how availability, pricing, and amenities
can be jointly analyzed to understand market dynamics and customer preferences
in a major urban market such as New York City.
